# 02 · Catálogo de cheias e chuva antecedente

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucaswsiviero/tubarao-flood/blob/main/notebooks/02_catalogo_cheias.ipynb)

**Objetivo:** transformar as séries diárias em uma **lista de eventos de cheia** (data, severidade em
tempo de retorno, estações atingidas) e juntar a **chuva que veio antes** de cada um. Essa tabela é a
base dos modelos de previsão das próximas fases.

> ⚠️ **Projeto experimental.** Nada aqui substitui os alertas oficiais da Defesa Civil,
> do SGB/CPRM e da ANA.

**Como funciona**
1. Para cada estação de vazão, ajusta-se a GEV aos máximos anuais (como no notebook 01).
2. Cada valor diário vira um **tempo de retorno equivalente (TR)**.
3. Um **evento** é um período em que alguma estação passa de um TR mínimo (padrão: 5 anos);
   picos separados por mais de 7 dias contam como eventos diferentes.
4. Para cada evento, calcula-se a chuva média da bacia nos dias anteriores ao pico.

## 1. Preparação

In [ ]:
import sys, subprocess
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/tubarao-flood/data")
else:
    DATA_DIR = Path("data")
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)

REPO = "https://github.com/lucaswsiviero/tubarao-flood.git"
PASTA = Path("tubarao-flood")
if PASTA.exists():
    try:
        subprocess.run(["git", "-C", str(PASTA), "pull", "--ff-only"], check=False)
    except OSError:
        pass
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(PASTA)], check=True)
sys.path.insert(0, str(PASTA / "src"))
print("Dados em:", DATA_DIR.resolve())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tubarao import dados, extremos
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
pd.options.display.float_format = "{:,.1f}".format
pd.options.display.width = 200

series = dados.carregar_todos(DATA_DIR)      # usa o cache do Drive; baixa só o que faltar

## 2. Tempo de retorno de cada dia

Ajustamos a GEV (momentos-L) aos máximos anuais de cada estação de vazão e convertemos **todos os dias**
em TR equivalente. Assim estações de rios e tamanhos diferentes ficam na mesma escala.

In [ ]:
VAZOES = [n for n in dados.VAZOES if n in series]
par = {n: extremos.gev_ajusta(extremos.maximos_anuais(series[n]).values) for n in VAZOES}

tr = pd.concat({n: pd.Series(extremos.tr_de_valores(series[n].valor.values, par[n]),
                             index=series[n].data) for n in VAZOES}, axis=1, sort=True)
tr = tr.asfreq("D")                                    # buracos de dado viram NaN
vazao = pd.concat({n: series[n].set_index("data").valor for n in VAZOES}, axis=1, sort=True).asfreq("D")

resumo = pd.DataFrame({n: {"início": tr[n].first_valid_index().date(), "fim": tr[n].last_valid_index().date(),
                           "k (forma)": f"{par[n][2]:+.2f}",
                           "vazão p/ TR=5 (m³/s)": extremos.gev_nivel(5, par[n]),
                           "vazão p/ TR=10 (m³/s)": extremos.gev_nivel(10, par[n])} for n in VAZOES}).T
resumo

## 3. Detectar eventos

`LIMIAR_TR` define o que é "cheia relevante". Com 5 anos, esperamos em média 1 evento a cada 5 anos
**por estação**, e como são 4 estações em períodos diferentes, a lista terá algumas dezenas de eventos.

In [ ]:
LIMIAR_TR = 5      # anos
FOLGA_DIAS = 7     # picos separados por mais que isso viram eventos diferentes
TR_CAP = 200       # com 30-80 anos de dados, TR acima disso não é informativo: mostramos "≥ 200"

tr_max = tr.max(axis=1, skipna=True)                   # pior TR entre as estações, por dia
n_est = tr.notna().sum(axis=1)                         # quantas estações tinham dado no dia

def agrupa_eventos(tr_max, limiar, folga):
    ev, atual = [], None
    for d, v in tr_max[tr_max >= limiar].items():
        if atual and (d - atual["fim"]).days <= folga:
            atual["fim"] = d
            if v > atual["tr_max"]:
                atual["pico"], atual["tr_max"] = d, v
        else:
            if atual:
                ev.append(atual)
            atual = {"inicio": d, "fim": d, "pico": d, "tr_max": v}
    if atual:
        ev.append(atual)
    return pd.DataFrame(ev)

ev = agrupa_eventos(tr_max, LIMIAR_TR, FOLGA_DIAS)
print(f"{len(ev)} eventos com TR >= {LIMIAR_TR} anos em pelo menos uma estação")

### 3.1 Enriquecer cada evento
Para cada evento: vazão de pico e TR em cada estação (na janela do evento ± 2 dias) e quantas
estações passaram do limiar (um evento em várias estações ao mesmo tempo é **regional**).

In [ ]:
linhas = []
for _, e in ev.iterrows():
    j = slice(e.inicio - pd.Timedelta(days=2), e.fim + pd.Timedelta(days=2))
    trj, vj = tr.loc[j], vazao.loc[j]
    pior = trj.max().idxmax()
    dias_ano = int((series[pior].data.dt.year == e.pico.year).sum())   # dias com dado, na estação do pico, naquele ano
    linha = {"pico": e.pico.date(), "duração (d)": (e.fim - e.inicio).days + 1,
             "TR máx (anos)": min(e.tr_max, TR_CAP), "estação do pico": pior.replace(" · vazão", ""),
             "estações c/ dado": int(trj.notna().any().sum()),
             "estações ≥ limiar": int((trj.max() >= LIMIAR_TR).sum()),
             "dias c/ dado no ano": dias_ano,
             "dado suspeito?": "SIM" if (dias_ano < 300 or e.tr_max > TR_CAP) else ""}
    for n in VAZOES:
        linha["Q " + n.split(" ·")[0].split(" (")[0]] = vj[n].max()
    linhas.append(linha)
catalogo = pd.DataFrame(linhas)
print("TR máx limitado a", TR_CAP, "anos. 'dado suspeito?' = ano com <300 dias de dado na estação do pico, ou TR acima do limite.")
print("Eventos suspeitos:", int((catalogo["dado suspeito?"] == "SIM").sum()), "de", len(catalogo))
catalogo.sort_values("TR máx (anos)", ascending=False).head(20).reset_index(drop=True)

## 4. Chuva antecedente

Chuva média diária da bacia = média das estações pluviométricas que tinham dado no dia
(mínimo 1; o número usado fica registrado em `n_pluv`). As janelas somam a chuva dos **dias que
terminam no pico**; a antecedente de 30 dias termina **7 dias antes** do pico, para medir o quanto o solo
já estava úmido.

In [ ]:
CHUVAS = [n for n in dados.CHUVAS if n in series]
chuva = pd.concat({n: series[n].set_index("data").valor for n in CHUVAS}, axis=1, sort=True).asfreq("D")
chuva_media = chuva.mean(axis=1, skipna=True)
n_pluv = chuva.notna().sum(axis=1)

r1, r3, r7 = (chuva_media.rolling(k, min_periods=k).sum() for k in (1, 3, 7))
ant30 = chuva_media.rolling(30, min_periods=25).sum().shift(7)

def feats(pico):
    p = pd.Timestamp(pico)
    janela = slice(p - pd.Timedelta(days=7), p)
    dia_chuva = chuva_media.loc[janela].idxmax() if chuva_media.loc[janela].notna().any() else pd.NaT
    return pd.Series({
        "chuva 1d (mm)": r1.get(p, np.nan),
        "chuva 3d (mm)": r3.loc[p - pd.Timedelta(days=3):p].max(),
        "chuva 7d (mm)": r7.get(p, np.nan),
        "antec. 30d (mm)": ant30.get(p, np.nan),
        "n_pluv": n_pluv.get(p, 0),
        "defasagem chuva→pico (d)": (p - dia_chuva).days if pd.notna(dia_chuva) else np.nan,
    })

catalogo = pd.concat([catalogo, catalogo["pico"].apply(feats)], axis=1)
mostrar = ["pico", "TR máx (anos)", "estação do pico", "estações ≥ limiar", "dado suspeito?",
           "chuva 1d (mm)", "chuva 3d (mm)", "chuva 7d (mm)", "antec. 30d (mm)", "n_pluv"]
catalogo.sort_values("TR máx (anos)", ascending=False)[mostrar].head(20).reset_index(drop=True)

In [ ]:
print("Eventos com dado de chuva:", int(catalogo["chuva 3d (mm)"].notna().sum()), "de", len(catalogo))
print()
print(catalogo[["chuva 1d (mm)", "chuva 3d (mm)", "chuva 7d (mm)", "antec. 30d (mm)"]].describe().round(1).to_string())
print()
print("Defasagem (dias) entre o dia de maior chuva e o pico de vazão:")
print(catalogo["defasagem chuva→pico (d)"].value_counts().sort_index().to_string())

## 5. Visualização

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
c = catalogo.dropna(subset=["chuva 3d (mm)"])
sc = axs[0].scatter(c["chuva 3d (mm)"], c["TR máx (anos)"], c=c["antec. 30d (mm)"], cmap="viridis", s=40, edgecolor="k", lw=.3)
axs[0].set_yscale("log"); axs[0].set_xlabel("chuva média da bacia em 3 dias (mm)"); axs[0].set_ylabel("TR máximo do evento (anos)")
axs[0].set_title("Quanto choveu × severidade da cheia"); plt.colorbar(sc, ax=axs[0], label="chuva dos 30 dias antes (mm)")

anos = pd.to_datetime(catalogo["pico"]).dt.year
decada = (anos // 10 * 10).value_counts().sort_index()
axs[1].bar(decada.index.astype(str) + "s", decada.values)
axs[1].set_title(f"Eventos com TR ≥ {LIMIAR_TR} anos por década"); axs[1].set_ylabel("nº de eventos")
plt.tight_layout(); plt.show()
print("Atenção: a contagem por década depende de quais estações tinham dado em cada época (ver seção 2).")

## 6. Primeiro "gatilho" de chuva e o prazo de aviso

Pergunta: **se a chuva média de 3 dias (observada até o dia *d*) passa de X mm, uma cheia começa logo em seguida?**
E o ponto central: **com quanta antecedência?** A seção 4 mostra a defasagem entre o dia de maior chuva e o
pico. Se ela for quase sempre 0–1 dia, a chuva *observada* só avisa em cima da hora.

Dois prazos são testados, com a mesma regra de "início de evento" (nenhuma estação acima do limiar antes, e
alguma passando dele na janela):
- **0–1 dia à frente**: o evento começa no próprio dia *d* ou no seguinte;
- **1–3 dias à frente**: o evento começa de *d+1* a *d+3* (aviso com antecedência real).

`precisão` = dos dias com alerta, quantos tiveram início de evento. `cobertura` = dos inícios de evento,
quantos tiveram alerta. Só entram dias de **1987 a 2021** com pelo menos 2 pluviômetros. Os dias são
correlacionados e os eventos são poucos: **tome como ordem de grandeza, não como regra operacional.**

In [ ]:
idx = pd.date_range("1987-01-01", "2021-12-31", freq="D")        # mesmo índice para todas as séries

def testa_gatilho(ini, fim, limiares=(30, 50, 70, 90, 110, 130)):
    """Início de evento na janela [d+ini, d+fim], dado que nenhuma estação estava acima do limiar em d+ini-1."""
    n = fim - ini + 1
    antes = tr_max.shift(-(ini - 1)) < LIMIAR_TR
    pior = tr_max.rolling(n, min_periods=1).max().shift(-fim)
    tem = tr_max.rolling(n, min_periods=1).count().shift(-fim) > 0
    d = pd.DataFrame({"r3": r3.reindex(idx), "npl": n_pluv.reindex(idx), "tr": tr_max.reindex(idx),
                      "ok": (antes & tem).reindex(idx).fillna(False).astype(bool),
                      "inicio": (antes & (pior >= LIMIAR_TR)).reindex(idx).fillna(False).astype(bool)})
    d = d[(d.npl >= 2) & d.r3.notna() & d.ok]
    linhas = []
    for thr in limiares:
        alerta = d.r3 >= thr
        ac = int((alerta & d.inicio).sum())
        linhas.append({"chuva 3d ≥ (mm)": thr, "dias c/ alerta": int(alerta.sum()), "acertos": ac,
                       "precisão": f"{ac / max(int(alerta.sum()), 1):.0%}",
                       "cobertura": f"{ac / max(int(d.inicio.sum()), 1):.0%}"})
    return len(d), int(d.inicio.sum()), pd.DataFrame(linhas).set_index("chuva 3d ≥ (mm)")

for ini, fim in [(0, 1), (1, 3)]:
    ndias, ninicios, tab = testa_gatilho(ini, fim)
    print(f"=== Evento começando de d+{ini} a d+{fim}: {ndias:,} dias avaliados, {ninicios} inícios de evento ===")
    display(tab)

## 7. Lacuna principal: Tubarão a partir de 2022

A série de cota da estação **Tubarão (84580500)** termina em dez/2021 no serviço público. O inventário da ANA
indica que a estação é telemétrica desde 2013, mas as operações de telemetria deste serviço **voltam
vazias**. Para as cheias de 2022 em diante na própria cidade, será preciso:
- solicitar acesso à **API nova (HidroWebService)** da ANA, que exige credenciais, ou
- obter os dados do **SGB/CPRM** (sistema de alerta da bacia) ou da **Defesa Civil**.

In [ ]:
for n in ("Tubarão · cota", "Pedras Grandes · vazão", "São Ludgero (B. Norte) · vazão"):
    if n in series:
        print(f"{n:34s} último dado: {series[n].data.max().date()}")

## 8. Salvar

In [ ]:
catalogo.to_csv(DATA_DIR / "processed" / "catalogo_cheias.csv", index=False)
tr_max.rename("tr_max").to_frame().join(chuva_media.rename("chuva_media_mm")).to_parquet(DATA_DIR / "processed" / "diario_tr_chuva.parquet")
print("Salvo em", (DATA_DIR / "processed").resolve())
print(*sorted(p.name for p in (DATA_DIR / "processed").iterdir()), sep="\n")

## Próximos passos
1. **Eventos antes de 1985** a partir de jornais e relatórios (a bacia tem cheias históricas que as
   estações automáticas não pegaram) e anexar ao catálogo.
2. **Chuva em grade** (CHIRPS/ERA5) em vez de média de poucos pluviômetros, para ver *onde* choveu na bacia.
3. **Índices de larga escala** (ENSO etc.) e umidade do solo como variáveis sazonais.
4. **Modelo de previsão** (vazão ou nível 6–48 h à frente) usando este catálogo como verdade de referência.